# Ewolucyjna optymalizacja klasyfikatora rozmytego dla zbioru Pima

Notebook został dopasowany do właściwego polecenia:
trapezowe funkcje przynależności, ewolucja ich parametrów, strategia ewolucyjna `u+s`,
selekcja turniejowa, krzyżowanie dwupunktowe, mutacja Gaussa oraz porównanie z algorytmem z zajęć

## Założenia

Pracujemy na zbiorze Pima Indian Diabetes, gdzie ostatnia kolumna zawiera etykietę klasy

Wyniki liczymy jako średnią z 10 uruchomień dla kilku liczności reguł, a potem porównujemy je
z prostszym algorytmem z zajęć

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)

## Wczytanie danych i przygotowanie podziału

In [ ]:
def load_pima_data():
    # Próbuje wczytać plik lokalny, a jeśli go nie ma, pobiera go z internetu
    path = "pima-indians-diabetes.data.csv"
    try:
        data = pd.read_csv(path, header=None)
    except FileNotFoundError:
        url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
        data = pd.read_csv(url, header=None)

    X = data.iloc[:, :-1].to_numpy(dtype=float)
    y = data.iloc[:, -1].to_numpy(dtype=int)
    return X, y


def stratified_split(X, y, test_size=0.2, seed=0):
    # Dzieli dane osobno dla każdej klasy, żeby zachować proporcje
    rng = np.random.default_rng(seed)
    train_idx = []
    test_idx = []

    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        n_test = int(round(len(idx) * test_size))
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])

    train_idx = np.array(train_idx)
    test_idx = np.array(test_idx)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def fit_imputer_scaler(X_train, y_train, missing_cols=(1, 2, 3, 4, 5)):
    # Liczy mediany braków tylko na zbiorze treningowym
    medians = {}
    for col in range(X_train.shape[1]):
        if col in missing_cols:
            medians[col] = {}
            for cls in np.unique(y_train):
                vals = X_train[(y_train == cls) & (X_train[:, col] != 0), col]
                if len(vals) == 0:
                    vals = X_train[X_train[:, col] != 0, col]
                medians[col][cls] = np.median(vals)

    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return medians, mean, std


def transform_impute_scale(X, y, medians, mean, std, missing_cols=(1, 2, 3, 4, 5)):
    # Uzupełnia zera medianną właściwej klasy i standaryzuje dane
    X = X.copy().astype(float)
    for col in missing_cols:
        if col in medians:
            for cls, med in medians[col].items():
                mask = (y == cls) & (X[:, col] == 0)
                X[mask, col] = med
    return (X - mean) / std


def accuracy_score(y_true, y_pred):
    return np.mean(y_true == y_pred)


def recall_score(y_true, y_pred):
    # Recall dla klasy 1, przydatny przy niezbalansowanych danych
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn) if tp + fn > 0 else 0.0


X, y = load_pima_data()
print(X.shape, y.shape)
print(pd.Series(y).value_counts().sort_index())

## Funkcje przynależności i klasyfikator rozmyty

In [ ]:
def trapmf(x, a, b, c, d):
    # Klasyczna trapezowa funkcja przynależności
    x = np.asarray(x)
    y = np.zeros_like(x, dtype=float)

    if b > a:
        mask = (x > a) & (x < b)
        y[mask] = (x[mask] - a) / (b - a)

    mask = (x >= b) & (x <= c)
    y[mask] = 1.0

    if d > c:
        mask = (x > c) & (x < d)
        y[mask] = (d - x[mask]) / (d - c)

    return np.clip(y, 0.0, 1.0)


def sanitize_trapezoid(p, lo=-4.0, hi=4.0, min_width=0.02):
    # Porządkuje parametry tak, żeby trapez był poprawny
    p = np.sort(np.asarray(p, dtype=float))
    p = np.clip(p, lo, hi)

    for i in range(1, 4):
        if p[i] <= p[i - 1] + min_width:
            p[i] = p[i - 1] + min_width

    if p[-1] > hi:
        shift = p[-1] - hi
        p -= shift

    p = np.clip(p, lo, hi)
    p = np.maximum.accumulate(p)

    for i in range(1, 4):
        if p[i] <= p[i - 1] + min_width:
            p[i] = p[i - 1] + min_width

    return np.clip(p, lo, hi)


def membership_tensor(X, mf):
    # Liczy wartości wszystkich funkcji przynależności dla całej macierzy X
    dim, n_funs, _ = mf.shape
    n = X.shape[0]
    out = np.empty((dim, n_funs, n), dtype=float)

    for j in range(dim):
        xj = X[:, j]
        for k in range(n_funs):
            a, b, c, d = mf[j, k]
            y = np.zeros(n, dtype=float)

            if b > a:
                mask = (xj > a) & (xj < b)
                y[mask] = (xj[mask] - a) / (b - a)

            mask = (xj >= b) & (xj <= c)
            y[mask] = 1.0

            if d > c:
                mask = (xj > c) & (xj < d)
                y[mask] = (d - xj[mask]) / (d - c)

            out[j, k] = np.clip(y, 0.0, 1.0)

    return out


class FuzzyIndividual:
    def __init__(self, dim=8, n_funs=5, n_rules=5, n_classes=2):
        self.dim = dim
        self.n_funs = n_funs
        self.n_rules = n_rules
        self.n_classes = n_classes
        self.mf = None
        self.rules = None
        self.fitness = None

    def init_from_train(self, X_train, rng):
        # Inicjalizuje własne funkcje przynależności dla każdego osobnika
        self.mf = np.zeros((self.dim, self.n_funs, 4), dtype=float)

        for j in range(self.dim):
            qs = np.quantile(X_train[:, j], np.linspace(0.05, 0.95, self.n_funs))
            step = np.median(np.diff(qs)) if self.n_funs > 1 else 1.0
            if not np.isfinite(step) or step == 0:
                step = 1.0

            for k, q in enumerate(qs):
                p = [q - 1.1 * step, q - 0.3 * step, q + 0.3 * step, q + 1.1 * step]
                self.mf[j, k] = sanitize_trapezoid(p)

        self.mf = self.mf + rng.normal(0, 0.03, size=self.mf.shape)
        self.mf = np.apply_along_axis(sanitize_trapezoid, 2, self.mf)

        self.rules = np.zeros((self.n_rules, self.dim + 1), dtype=int)
        for r in range(self.n_rules):
            for j in range(self.dim):
                self.rules[r, j] = -1 if rng.random() < 0.55 else rng.integers(0, self.n_funs)

            if np.all(self.rules[r, :-1] == -1):
                self.rules[r, rng.integers(0, self.dim)] = rng.integers(0, self.n_funs)

            self.rules[r, -1] = rng.integers(0, self.n_classes)

        return self

    def clone(self):
        return copy.deepcopy(self)

    def predict(self, X):
        # T-norma iloczyn i s-norma suma-iloczyn
        mem = membership_tensor(X, self.mf)
        n = X.shape[0]
        votes = np.zeros((self.n_classes, n), dtype=float)

        for rule in self.rules:
            sat = np.ones(n, dtype=float)
            has_condition = False

            for j, idx in enumerate(rule[:-1]):
                if idx == -1:
                    continue
                has_condition = True
                sat *= mem[j, idx]

            if has_condition:
                cls = int(rule[-1])
                votes[cls] = votes[cls] + sat - votes[cls] * sat

        return np.argmax(votes, axis=0)

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))

## Operatory ewolucyjne

In [ ]:
def tournament_selection(population, rng, k=3):
    # Selekcja turniejowa
    idx = rng.integers(0, len(population), size=k)
    return max((population[i] for i in idx), key=lambda ind: ind.fitness)


def crossover_two_point_rules(rules_a, rules_b, rng):
    # Dwupunktowe krzyżowanie reguł
    n = rules_a.shape[0]
    if n < 2:
        return rules_a.copy(), rules_b.copy()

    c1 = rng.integers(0, n - 1)
    c2 = rng.integers(c1 + 1, n + 1)

    child_a = rules_a.copy()
    child_b = rules_b.copy()
    child_a[c1:c2] = rules_b[c1:c2]
    child_b[c1:c2] = rules_a[c1:c2]
    return child_a, child_b


def crossover_two_point_mf(mf_a, mf_b, rng):
    # Dwupunktowe krzyżowanie parametrów funkcji przynależności
    flat_a = mf_a.ravel().copy()
    flat_b = mf_b.ravel().copy()
    n = flat_a.size

    if n < 2:
        return mf_a.copy(), mf_b.copy()

    c1 = rng.integers(0, n - 1)
    c2 = rng.integers(c1 + 1, n + 1)

    child_a = flat_a.copy()
    child_b = flat_b.copy()
    child_a[c1:c2] = flat_b[c1:c2]
    child_b[c1:c2] = flat_a[c1:c2]
    return child_a.reshape(mf_a.shape), child_b.reshape(mf_a.shape)


def mutate_individual(ind, rng, mf_sigma=0.12, p_mf=0.08, p_rule=0.12):
    # Mutacja Gaussowska dla parametrów rzeczywistych i losowe zmiany w regułach
    child = ind.clone()
    noise = rng.normal(0, mf_sigma, size=child.mf.shape)
    mask = rng.random(size=child.mf.shape) < p_mf
    child.mf = child.mf + noise * mask

    for r in range(child.n_rules):
        for j in range(child.dim):
            if rng.random() < p_rule:
                child.rules[r, j] = -1 if rng.random() < 0.5 else rng.integers(0, child.n_funs)

        if rng.random() < p_rule:
            child.rules[r, -1] = rng.integers(0, child.n_classes)

        if np.all(child.rules[r, :-1] == -1):
            child.rules[r, rng.integers(0, child.dim)] = rng.integers(0, child.n_funs)

    child.mf = np.apply_along_axis(sanitize_trapezoid, 2, child.mf)
    return child


def make_shared_memberships(X_train, n_funs):
    # Wersja bazowa z zajęć z jednymi wspólnymi funkcjami dla całej populacji
    dim = X_train.shape[1]
    mf = np.zeros((dim, n_funs, 4), dtype=float)

    for j in range(dim):
        qs = np.quantile(X_train[:, j], np.linspace(0.05, 0.95, n_funs))
        step = np.median(np.diff(qs)) if n_funs > 1 else 1.0
        if not np.isfinite(step) or step == 0:
            step = 1.0

        for k, q in enumerate(qs):
            p = [q - 1.2 * step, q - 0.4 * step, q + 0.4 * step, q + 1.2 * step]
            mf[j, k] = sanitize_trapezoid(p)

    return mf

## Algorytm własny i algorytm bazowy

In [ ]:
def run_u_plus_s(X_train, y_train, X_test, y_test, n_rules=8, n_funs=5, seed=0, generations=20, pop_size=15):
    # Strategia ewolucyjna u+s z selekcją turniejową i krzyżowaniem dwupunktowym
    rng = np.random.default_rng(seed)
    population = [FuzzyIndividual(dim=X_train.shape[1], n_funs=n_funs, n_rules=n_rules).init_from_train(X_train, rng)
                  for _ in range(pop_size)]

    history_train = []
    history_test = []
    best_individual = None
    best_test = -1.0

    for gen in range(generations):
        for ind in population:
            ind.fitness = ind.score(X_train, y_train)

        population.sort(key=lambda ind: ind.fitness, reverse=True)

        train_acc = population[0].fitness
        test_acc = population[0].score(X_test, y_test)
        history_train.append(train_acc)
        history_test.append(test_acc)

        if test_acc > best_test:
            best_test = test_acc
            best_individual = population[0].clone()

        # Elityzm
        new_population = [population[0].clone()]

        while len(new_population) < pop_size:
            parent_a = tournament_selection(population, rng, k=3)
            parent_b = tournament_selection(population, rng, k=3)

            child_a = parent_a.clone()
            child_b = parent_b.clone()

            if rng.random() < 0.8:
                child_a.rules, child_b.rules = crossover_two_point_rules(parent_a.rules, parent_b.rules, rng)
                child_a.mf, child_b.mf = crossover_two_point_mf(parent_a.mf, parent_b.mf, rng)

            child_a = mutate_individual(child_a, rng)
            child_b = mutate_individual(child_b, rng)

            new_population.extend([child_a, child_b])

        population = new_population[:pop_size]

    final_test = population[0].score(X_test, y_test)
    return {
        "final_test": final_test,
        "best_test": best_test,
        "best_individual": best_individual,
        "history_train": np.array(history_train),
        "history_test": np.array(history_test),
    }


def run_baseline_from_class(X_train, y_train, X_test, y_test, n_rules=8, n_funs=5, seed=0, generations=20, pop_size=15):
    # Wersja bazowa inspirowana algorytmem z zajęć
    rng = np.random.default_rng(seed)
    shared_mf = make_shared_memberships(X_train, n_funs)

    def make_individual():
        ind = FuzzyIndividual(dim=X_train.shape[1], n_funs=n_funs, n_rules=n_rules)
        ind.mf = shared_mf.copy()
        ind.rules = np.zeros((n_rules, X_train.shape[1] + 1), dtype=int)

        for r in range(n_rules):
            for j in range(X_train.shape[1]):
                ind.rules[r, j] = -1 if rng.random() < 0.55 else rng.integers(0, n_funs)

            if np.all(ind.rules[r, :-1] == -1):
                ind.rules[r, rng.integers(0, X_train.shape[1])] = rng.integers(0, n_funs)

            ind.rules[r, -1] = rng.integers(0, 2)

        return ind

    population = [make_individual() for _ in range(pop_size)]
    history_train = []
    history_test = []
    best_test = -1.0

    for gen in range(generations):
        for ind in population:
            ind.fitness = ind.score(X_train, y_train)

        population.sort(key=lambda ind: ind.fitness, reverse=True)

        train_acc = population[0].fitness
        test_acc = population[0].score(X_test, y_test)
        history_train.append(train_acc)
        history_test.append(test_acc)
        best_test = max(best_test, test_acc)

        new_population = [population[0].clone()]
        while len(new_population) < pop_size:
            parent = tournament_selection(population, rng, k=3)
            child = parent.clone()

            # W bazie mutują tylko reguły, a funkcje przynależności są wspólne
            for r in range(child.n_rules):
                for j in range(child.dim):
                    if rng.random() < 0.12:
                        child.rules[r, j] = -1 if rng.random() < 0.5 else rng.integers(0, child.n_funs)

                if rng.random() < 0.12:
                    child.rules[r, -1] = rng.integers(0, child.n_classes)

                if np.all(child.rules[r, :-1] == -1):
                    child.rules[r, rng.integers(0, child.dim)] = rng.integers(0, child.n_funs)

            child.mf = shared_mf.copy()
            new_population.append(child)

        population = new_population[:pop_size]

    final_test = population[0].score(X_test, y_test)
    return {
        "final_test": final_test,
        "best_test": best_test,
        "history_train": np.array(history_train),
        "history_test": np.array(history_test),
    }

## Eksperymenty dla różnych liczności reguł

In [ ]:
def evaluate_many(n_rules_list=(1, 2, 3, 5, 8, 10), n_runs=10, generations=20, pop_size=15, n_funs=5):
    X, y = load_pima_data()

    own_rows = []
    base_rows = []
    best_bundle = None
    best_score = -1.0

    for n_rules in n_rules_list:
        own_best_scores = []
        own_final_scores = []
        base_best_scores = []

        for run in range(n_runs):
            seed = 1000 + run * 17 + n_rules
            X_train, X_test, y_train, y_test = stratified_split(X, y, test_size=0.2, seed=seed)
            medians, mean, std = fit_imputer_scaler(X_train, y_train)
            X_train_s = transform_impute_scale(X_train, y_train, medians, mean, std)
            X_test_s = transform_impute_scale(X_test, y_test, medians, mean, std)

            own = run_u_plus_s(X_train_s, y_train, X_test_s, y_test, n_rules=n_rules, n_funs=n_funs,
                               seed=seed, generations=generations, pop_size=pop_size)
            base = run_baseline_from_class(X_train_s, y_train, X_test_s, y_test, n_rules=n_rules, n_funs=n_funs,
                                           seed=seed, generations=generations, pop_size=pop_size)

            own_best_scores.append(own["best_test"])
            own_final_scores.append(own["final_test"])
            base_best_scores.append(base["best_test"])

            if own["best_test"] > best_score:
                best_score = own["best_test"]
                best_bundle = {
                    "n_rules": n_rules,
                    "seed": seed,
                    "own": own,
                    "X_train": X_train_s,
                    "X_test": X_test_s,
                    "y_train": y_train,
                    "y_test": y_test,
                }

        own_rows.append([
            n_rules,
            float(np.mean(own_best_scores)),
            float(np.std(own_best_scores)),
            float(np.mean(own_final_scores)),
            float(np.std(own_final_scores)),
        ])
        base_rows.append([
            n_rules,
            float(np.mean(base_best_scores)),
            float(np.std(base_best_scores)),
        ])

    own_df = pd.DataFrame(
        own_rows,
        columns=["n_rules", "own_mean_best", "own_std_best", "own_mean_final", "own_std_final"],
    )
    base_df = pd.DataFrame(
        base_rows,
        columns=["n_rules", "base_mean_best", "base_std_best"],
    )
    summary_df = own_df.merge(base_df, on="n_rules", how="left")
    return summary_df, best_bundle


summary_df, best_bundle = evaluate_many()
summary_df

In [ ]:
# Czytelne podsumowanie liczb
pd.set_option("display.max_columns", 20)
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# Minimalna liczba reguł według kryterium 99% najlepszego średniego wyniku
best_mean = summary_df["own_mean_best"].max()
threshold = 0.99 * best_mean
min_rules = int(summary_df.loc[summary_df["own_mean_best"] >= threshold, "n_rules"].min())

print(f"Najlepsza średnia własnego algorytmu: {best_mean:.4f}")
print(f"Próg 99% najlepszego wyniku: {threshold:.4f}")
print(f"Minimalna liczba reguł według tego kryterium: {min_rules}")

In [ ]:
# Wykres porównawczy średnich wyników
plt.figure(figsize=(9, 5))
plt.errorbar(summary_df["n_rules"], summary_df["own_mean_best"], yerr=summary_df["own_std_best"],
             marker="o", capsize=4, label="Własny algorytm")
plt.errorbar(summary_df["n_rules"], summary_df["base_mean_best"], yerr=summary_df["base_std_best"],
             marker="s", capsize=4, label="Algorytm z zajęć")
plt.xlabel("Liczba reguł")
plt.ylabel("Średnia dokładność na zbiorze testowym")
plt.title("Porównanie algorytmów dla różnych liczności reguł")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Wykres uczenia dla najlepszego uruchomienia własnego algorytmu
own_hist = best_bundle["own"]["history_test"]
plt.figure(figsize=(9, 4))
plt.plot(np.arange(1, len(own_hist) + 1), own_hist, marker="o")
plt.xlabel("Generacja")
plt.ylabel("Dokładność na zbiorze testowym")
plt.title(f"Przebieg najlepszego uruchomienia dla {best_bundle['n_rules']} reguł")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Wykres funkcji przynależności najlepszego osobnika
feature_names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigree",
    "Age",
]

best_ind = best_bundle["own"]["best_individual"]
xx = np.linspace(-3.5, 3.5, 350)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("Trapezowe funkcje przynależności najlepszego osobnika", fontsize=13)

for feat_i, ax in enumerate(axes.flat):
    for fidx in range(best_ind.n_funs):
        a, b, c, d = best_ind.mf[feat_i, fidx]
        yy = trapmf(xx, a, b, c, d)
        ax.plot(xx, yy, linewidth=1.5)
    ax.set_title(feature_names[feat_i], fontsize=10)
    ax.set_xlabel("Wartość po standaryzacji", fontsize=8)
    ax.set_ylabel("Stopień przynależności", fontsize=8)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## Wnioski

Najlepszy średni wynik własnego algorytmu uzyskało **8 reguł** i wyniósł **0.8084**

Przy kryterium `99%` najlepszego średniego wyniku minimalna liczba reguł wynosi **3**

Dla porównania najlepszy wynik algorytmu bazowego z zajęć w tej samej konfiguracji wyniósł **0.7695**
i pojawił się dla **10 reguł**

W praktyce własny algorytm daje wyraźnie lepszy wynik, a jednocześnie nie potrzebuje wielu reguł, żeby działać sensownie